# HB model: follow-up fit for flagged subjects (widened bounds)

Picks up from the full 12-subject sweep (see `kaggle_hb_multisubject_fit.ipynb`).
Six subjects hit a parameter ceiling in that run, meaning their true best-fit value
may have been artificially capped rather than genuinely found:

| Subject | What hit its ceiling | Old bound |
|---|---|---|
| 2  | `k_motor = 399.0` | 400 |
| 5  | `k_prior[80]=996.7`, `k_prior[40]=1000.0` | 1000 |
| 6  | `k_prior[80]=997.2` | 1000 |
| 7  | `k_prior[80]=999.6` | 1000 |
| 10 | `alpha=0.010` (floor, not ceiling) | 0.01 |
| 11 | `k_prior[40]=999.5` | 1000 |

This notebook re-fits just these 6 with more headroom:
- `k_prior` ceiling: 1000 -> **5000**
- `alpha` floor: 0.01 -> **0.001**
- `k_motor` ceiling: 400 -> **1000**

Widening bounds for all 6 is harmless for whichever ones don't actually need it,
Powell will just re-find the same optimum it found before if the true value was
already safely inside the old bounds.

**Before running:** turn Internet ON if you don't already have the CSV uploaded as
an input from before (Settings > Internet), or just re-attach the CSV as an Input
like last time, no internet needed that way.


In [ ]:
import os
import time
import pickle
import glob
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
from scipy.special import i0e
from scipy.optimize import minimize

OUT_DIR = Path("/kaggle/working/hb_fits_followup")
if not OUT_DIR.parent.exists():
    OUT_DIR = Path("./hb_fits_followup")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints will be saved to: {OUT_DIR.resolve()}")


## 1. Data loading (same as before, uploaded input preferred over git clone)

In [ ]:
candidates = glob.glob("/kaggle/input/**/data01_direction4priors.csv", recursive=True)
candidates += glob.glob("./**/data01_direction4priors.csv", recursive=True)

if candidates:
    CSV_PATH = Path(candidates[0])
    print(f"Found uploaded file at: {CSV_PATH}")
else:
    print("No uploaded copy found, trying to clone the public repo (needs Internet ON)...")
    REPO_URL = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
    REPO_DIR = Path("/kaggle/working/NeuroMatch_2026_Behaviour")
    if not REPO_DIR.parent.exists():
        REPO_DIR = Path("./NeuroMatch_2026_Behaviour")
    if not REPO_DIR.exists():
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    matches = list(REPO_DIR.rglob("data01_direction4priors.csv"))
    if not matches:
        raise FileNotFoundError(
            "data01_direction4priors.csv not found locally and the git clone didn't work. "
            "Easiest fix: upload the CSV directly via Add Input > Upload."
        )
    CSV_PATH = matches[0]

raw = pd.read_csv(CSV_PATH)
print(f"Loaded from: {CSV_PATH}")
print(f"raw shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")


In [ ]:
data = raw.copy()

data['estimate_deg'] = np.degrees(np.arctan2(data.estimate_y, data.estimate_x)) % 360
data.loc[data.estimate_deg == 0, 'estimate_deg'] = 360

n_before = len(data)
data = data.dropna(subset=['estimate_deg', 'motion_direction']).copy()
print(f"dropped {n_before - len(data)} rows with missing estimate")

data['error_deg'] = ((data.estimate_deg - data.motion_direction + 180) % 360) - 180
data['block_id'] = data['subject_id'].astype(str) + '_' + data['run_id'].astype(str)

trial_data = data.rename(columns={
    'subject_id': 'subject',
    'trial_index': 'trial',
    'prior_std': 'prior_width',
    'motion_coherence': 'coherence',
    'motion_direction': 'true_direction',
})[['subject', 'block_id', 'trial', 'prior_width', 'coherence',
    'true_direction', 'estimate_deg', 'error_deg',
    'session_id', 'experiment_id']]

block_context = (
    data.groupby(['subject_id', 'run_id'], as_index=False)
    .agg(session_id=('session_id', 'first'))
    .sort_values(['subject_id', 'run_id'])
    .reset_index(drop=True)
)
block_context['prev_session_id'] = block_context.groupby('subject_id')['session_id'].shift(1)
block_context['same_session_prev'] = (
    block_context['prev_session_id'].notna()
    & (block_context['session_id'] == block_context['prev_session_id'])
)
block_context['block_id'] = block_context['subject_id'].astype(str) + '_' + block_context['run_id'].astype(str)

trial_data = trial_data.merge(
    block_context[['block_id', 'run_id', 'same_session_prev']], on='block_id', how='left'
)

FLAGGED_SUBJECTS = [2, 5, 6, 7, 10, 11]
print(f"trial_data shape: {trial_data.shape}")
print(f"This notebook only fits: {FLAGGED_SUBJECTS}")


## 2. The reliability-mixture model and its NLL (unchanged)

In [ ]:
DEG = np.arange(1, 361)

def vm_pdf(support_deg, mu_deg, k, norm=True):
    mu = np.atleast_1d(np.asarray(mu_deg, float))
    x = np.deg2rad(np.asarray(support_deg, float))[:, None]
    u = np.deg2rad(mu)[None, :]
    k = float(k)
    if np.isinf(k) or k > 1e300:
        out = np.zeros((len(support_deg), len(mu)))
        for j, mm in enumerate(mu):
            out[np.argmin(np.abs(np.asarray(support_deg) - mm)), j] = 1.0
    else:
        out = np.exp(k * np.cos(x - u) - k) / (2 * np.pi * i0e(k))
    if norm:
        out = out / out.sum(0, keepdims=True)
    return out


def wrap_signed_deg(diff_deg):
    return ((diff_deg + 180) % 360) - 180


def circular_mean_deg(angles_deg):
    angles_rad = np.deg2rad(np.asarray(angles_deg))
    mean_x = np.mean(np.cos(angles_rad))
    mean_y = np.mean(np.sin(angles_rad))
    return float(np.degrees(np.arctan2(mean_y, mean_x)) % 360)


def prior_agreement(measurement_deg, prior_mean_deg, k_prior):
    delta_rad = np.deg2rad(wrap_signed_deg(measurement_deg - prior_mean_deg))
    return float(np.exp(k_prior * (np.cos(delta_rad) - 1.0)))


def trial_percept_distribution(prior_mean_deg, measurement_deg, k_prior, k_llh, prior_reliance):
    prior_component = vm_pdf(DEG, prior_mean_deg, k_prior)[:, 0]
    llh_component = vm_pdf(DEG, measurement_deg, k_llh)[:, 0]
    mixture = prior_reliance * prior_component + (1 - prior_reliance) * llh_component
    return mixture / mixture.sum()


def circ_convolve_vec(p, kernel):
    return np.real(np.fft.ifft(np.fft.fft(p) * np.fft.fft(kernel)))


def hb_mixture_trial_loop(d, params, prior_mean_deg=225.0, window_size=5):
    k_llh, k_prior = params['k_llh'], params['k_prior']
    alpha, k_motor, lapse_rate = params['alpha'], params['k_motor'], params['lapse_rate']

    reliance = 0.5
    window_buf = deque(maxlen=window_size)
    prev_block_id = None
    motor_kernel = vm_pdf(DEG, 360.0, k_motor)[:, 0]

    n = len(d)
    p_obs = np.empty(n)
    reliance_trace = np.empty(n)

    block_ids = d['block_id'].to_numpy()
    same_sess = d['same_session_prev'].to_numpy()
    true_dirs = d['true_direction'].to_numpy(dtype=float)
    coherences = d['coherence'].to_numpy()
    prior_widths = d['prior_width'].to_numpy()
    est_degs = d['estimate_deg'].to_numpy(dtype=float)

    for i in range(n):
        is_new_block = block_ids[i] != prev_block_id
        if is_new_block and not bool(same_sess[i]):
            window_buf.clear()

        reliance_trace[i] = reliance
        kp = k_prior[prior_widths[i]]
        kl = k_llh[coherences[i]]

        mixture = trial_percept_distribution(prior_mean_deg, true_dirs[i], kp, kl, reliance)
        with_motor = circ_convolve_vec(mixture, motor_kernel)
        with_motor = np.clip(with_motor, 0, None)
        with_motor = with_motor / with_motor.sum()
        final = (1 - lapse_rate) * with_motor + lapse_rate * (1.0 / 360)

        e_idx = int(round(est_degs[i])) % 360
        if e_idx == 0:
            e_idx = 360
        p_obs[i] = final[e_idx - 1]

        window_buf.append(true_dirs[i])
        smoothed = circular_mean_deg(list(window_buf))
        agreement = prior_agreement(smoothed, prior_mean_deg, kp)
        reliance = float(np.clip(reliance + alpha * (agreement - reliance), 1e-4, 1 - 1e-4))

        prev_block_id = block_ids[i]

    return p_obs, reliance_trace


def hb_mixture_nll(theta, d, cohs_u, ps_u, window_size=5):
    i = 0
    k_llh = {c: float(np.exp(theta[i + j])) for j, c in enumerate(cohs_u)}; i += len(cohs_u)
    k_prior = {p: float(np.exp(theta[i + j])) for j, p in enumerate(ps_u)}; i += len(ps_u)
    alpha = 1.0 / (1.0 + np.exp(-theta[i])); i += 1
    k_motor = float(np.exp(theta[i])); i += 1
    lapse_rate = 0.3 / (1.0 + np.exp(-theta[i]))
    params = dict(k_llh=k_llh, k_prior=k_prior, alpha=alpha, k_motor=k_motor, lapse_rate=lapse_rate)
    p_obs, _ = hb_mixture_trial_loop(d, params, window_size=window_size)
    p_obs = np.clip(p_obs, 1e-320, None)
    return -np.log(p_obs).sum()


cohs_u, ps_u = [0.06, 0.12, 0.24], [80, 40, 20, 10]


## 3. The bounded 8-start fit, with widened bounds

Same 8 starting points and optimizer settings as the main sweep, but:
- `K_PRIOR_UPPER`: 1000 -> 5000
- `ALPHA_LOWER`: 0.01 -> 0.001
- `K_MOTOR_UPPER`: 400 -> 1000


In [ ]:
K_PRIOR_UPPER = 5000.0
ALPHA_LOWER = 0.001
K_MOTOR_UPPER = 1000.0
N_STARTS = 8
SEED = 42

def make_starts(seed=SEED, n=N_STARTS):
    rng = np.random.default_rng(seed)
    base_k_llh = np.array([1.0, 3.0, 8.0])
    base_k_prior = np.array([0.7, 2.8, 8.7, 33.3])
    base_alpha, base_k_motor, base_lapse = 0.1, 30.0, 0.05

    starts = [(base_k_llh.copy(), base_k_prior.copy(), base_alpha, base_k_motor, base_lapse)]
    for _ in range(n - 1):
        factor_llh = rng.choice([0.2, 0.5, 2.0, 5.0], size=3)
        factor_prior = rng.choice([0.2, 0.5, 2.0, 5.0], size=4)
        starts.append((
            base_k_llh * factor_llh, base_k_prior * factor_prior,
            float(rng.uniform(0.02, 0.4)), float(rng.uniform(5, 80)), float(rng.uniform(0.01, 0.15))
        ))
    return starts


def to_theta(k_llh, k_prior, alpha, k_motor, lapse):
    return np.concatenate([np.log(k_llh), np.log(k_prior),
                            [np.log(alpha / (1 - alpha))], [np.log(k_motor)],
                            [np.log(lapse / (0.3 - lapse))]])


def decode_theta(x):
    k_llh = np.exp(x[0:3])
    k_prior = np.exp(x[3:7])
    alpha = 1 / (1 + np.exp(-x[7]))
    k_motor = np.exp(x[8])
    lapse = 0.3 / (1 + np.exp(-x[9]))
    return k_llh, k_prior, alpha, k_motor, lapse


def bounds_widened():
    return (
        [(np.log(0.05), np.log(150))] * 3
        + [(np.log(0.05), np.log(K_PRIOR_UPPER))] * 4
        + [(np.log(ALPHA_LOWER / (1 - ALPHA_LOWER)), np.log(0.6 / 0.4))]
        + [(np.log(1), np.log(K_MOTOR_UPPER))]
        + [(np.log(0.001 / 0.299), np.log(0.25 / 0.05))]
    )


def fit_subject_bounded(subject_id, d_subject, n_starts=N_STARTS, seed=SEED):
    def obj(theta):
        return hb_mixture_nll(theta, d_subject, cohs_u, ps_u)

    starts = make_starts(seed=seed, n=n_starts)
    bounds = bounds_widened()
    results = []
    for start_id, (k_llh0, k_prior0, alpha0, k_motor0, lapse0) in enumerate(starts, 1):
        k_llh0 = np.clip(k_llh0, 0.05, 150)
        k_prior0 = np.clip(k_prior0, 0.05, K_PRIOR_UPPER)
        alpha0 = float(np.clip(alpha0, ALPHA_LOWER, 0.6))
        k_motor0 = float(np.clip(k_motor0, 1, K_MOTOR_UPPER))
        lapse0 = float(np.clip(lapse0, 0.001, 0.25))
        theta0 = to_theta(k_llh0, k_prior0, alpha0, k_motor0, lapse0)
        theta0 = np.clip(theta0, [b[0] for b in bounds], [b[1] for b in bounds])
        t0 = time.time()
        res = minimize(obj, theta0, method="Powell", bounds=bounds,
                       options=dict(maxiter=60, xtol=5e-3, ftol=5e-3))
        dt = time.time() - t0
        results.append((start_id, bool(res.success), float(res.fun), res.x.copy()))
        print(f"  subject {subject_id} | start {start_id:2d}: success={res.success} "
              f"NLL={res.fun:.1f}  ({dt:.0f}s)")
    return sorted(results, key=lambda r: r[2])


## 4a. Recover checkpoints from a previous commit (chunked runs)

Same pattern as the main sweep: after each commit, publish its Output as a new
Kaggle dataset and attach it as Input to the next one, and this cell will pick
up anything already fit.


In [ ]:
prior_checkpoints = glob.glob("/kaggle/input/**/subject_*.pkl", recursive=True)
recovered = 0
for p in prior_checkpoints:
    dest = OUT_DIR / Path(p).name
    if not dest.exists():
        import shutil
        shutil.copy(p, dest)
        recovered += 1
print(f"Recovered {recovered} checkpoint(s) from previously attached input dataset(s).")
already_done = sorted(int(p.stem.split('_')[1]) for p in OUT_DIR.glob("subject_*.pkl"))
print(f"Subjects with an existing checkpoint right now: {already_done}")


## 4b. Choose which subjects this commit will fit

Balanced into 3 chunks of 2 (largest paired with smallest by trial count),
each roughly ~1.9-2 hours:

| Chunk | Subjects | Total trials |
|---|---|---|
| 1 | `[2, 5]`  | 13,666 |
| 2 | `[6, 7]`  | 13,350 |
| 3 | `[11, 10]` | 12,518 |


In [ ]:
# --- EDIT THIS before each commit ---
SUBJECTS_TO_RUN = [2, 5]

print(f"This commit will attempt to fit: {SUBJECTS_TO_RUN}")


## 4c. Main loop: fit `SUBJECTS_TO_RUN`, with checkpointing

In [ ]:
def checkpoint_path(subject_id):
    return OUT_DIR / f"subject_{subject_id}.pkl"


for subject_id in SUBJECTS_TO_RUN:
    ckpt = checkpoint_path(subject_id)
    if ckpt.exists():
        print(f"subject {subject_id}: checkpoint already exists, skipping")
        continue

    print(f"\n=== fitting subject {subject_id} ===")
    d_subject = trial_data[trial_data.subject == subject_id].sort_values(['run_id', 'trial']).reset_index(drop=True)
    print(f"  {len(d_subject)} trials")

    t_start = time.time()
    results = fit_subject_bounded(subject_id, d_subject)
    elapsed = time.time() - t_start

    best_start_id, best_success, best_nll, best_theta = results[0]
    k_llh, k_prior, alpha, k_motor, lapse = decode_theta(best_theta)

    near_ceiling = bool(np.any(k_prior > 0.9 * K_PRIOR_UPPER) or (k_motor > 0.9 * K_MOTOR_UPPER))
    likely_degenerate = bool(alpha < 2 * ALPHA_LOWER)

    params_best_named = dict(
        k_llh={c: float(v) for c, v in zip(cohs_u, k_llh)},
        k_prior={p: float(v) for p, v in zip(ps_u, k_prior)},
        alpha=alpha, k_motor=k_motor, lapse_rate=lapse,
    )
    p_obs, reliance_trace = hb_mixture_trial_loop(d_subject, params_best_named)

    checkpoint = dict(
        subject_id=subject_id,
        n_trials=len(d_subject),
        all_starts=results,
        best_start_id=best_start_id,
        best_success=best_success,
        best_nll=best_nll,
        best_params=params_best_named,
        near_ceiling=near_ceiling,
        likely_degenerate=likely_degenerate,
        k_prior_upper_used=K_PRIOR_UPPER,
        alpha_lower_used=ALPHA_LOWER,
        k_motor_upper_used=K_MOTOR_UPPER,
        p_obs=p_obs,
        reliance_trace=reliance_trace,
        fit_time_seconds=elapsed,
    )
    with open(ckpt, "wb") as f:
        pickle.dump(checkpoint, f)

    flag = ""
    if near_ceiling:
        flag += "  [WARNING: still near a ceiling even with widened bounds]"
    if likely_degenerate:
        flag += "  [WARNING: still near the alpha floor even when lowered]"
    print(f"  done in {elapsed:.0f}s | best NLL={best_nll:.1f} | alpha={alpha:.4f}{flag}")

print("\nAll requested subjects processed (or already checkpointed).")


## 5. Summary across the flagged subjects

In [ ]:
rows = []
for ckpt_file in sorted(OUT_DIR.glob("subject_*.pkl")):
    with open(ckpt_file, "rb") as f:
        c = pickle.load(f)
    kp = c['best_params']['k_prior']
    kl = c['best_params']['k_llh']
    rows.append(dict(
        subject=c['subject_id'], n_trials=c['n_trials'], best_nll=c['best_nll'],
        alpha=c['best_params']['alpha'], k_motor=c['best_params']['k_motor'],
        lapse_rate=c['best_params']['lapse_rate'],
        k_prior_80=kp[80], k_prior_40=kp[40], k_prior_20=kp[20], k_prior_10=kp[10],
        near_ceiling=c['near_ceiling'], likely_degenerate=c['likely_degenerate'],
        fit_time_seconds=c['fit_time_seconds'],
    ))

summary = pd.DataFrame(rows).sort_values('subject').reset_index(drop=True)
summary_path = OUT_DIR / "summary_followup.csv"
summary.to_csv(summary_path, index=False)
print(f"saved summary to {summary_path}")
summary
